# FloodLens — Phase 2: Training a Real Baseline
### Train/val split, Dice+BCE loss, checkpointing to Drive, real IoU score

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

This notebook is self-contained (re-downloads FloodNet and rebuilds the dataset
pipeline from Phase 1) so it works in a fresh Colab session. If you already have a
Phase 1 session running with `dataset` in memory, you can skip straight to Section 4.


## 0. Setup

In [ ]:
!pip install -q kagglehub segmentation-models-pytorch albumentations opencv-python-headless
print("Dependencies installed.")


In [ ]:
import os
import re
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import kagglehub
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("WARNING: No GPU detected. Training will be slow. Check Runtime > Change runtime type > GPU.")


## 1. Mount Google Drive for checkpoint persistence
**This matters on free tier.** Colab sessions disconnect after ~90 min idle or ~12hr
max, and `/content` is wiped when that happens. Without this, a disconnect mid-training
loses everything. Checkpoints go to a `FloodLens_checkpoints` folder in your Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/FloodLens_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", CHECKPOINT_DIR)


## 2. Rebuild the data pipeline (from Phase 1)

In [ ]:
path = kagglehub.dataset_download("aletbm/aerial-imagery-dataset-floodnet-challenge")
print("Path to dataset files:", path)


In [ ]:
def extract_id(filename):
    match = re.match(r"(\d+)", filename)
    return match.group(1) if match else None

candidate_img_dirs = []
candidate_mask_dirs = []

for root, dirs, files in os.walk(path):
    name = os.path.basename(root).lower()
    if not files:
        continue
    if any(k in name for k in ["label", "mask", "gt", "annotation"]):
        candidate_mask_dirs.append(root)
    elif any(k in name for k in ["img", "image"]):
        candidate_img_dirs.append(root)

best_pair = None
best_overlap = -1
for img_d in candidate_img_dirs:
    img_ids = set(filter(None, (extract_id(f) for f in os.listdir(img_d))))
    for mask_d in candidate_mask_dirs:
        mask_ids = set(filter(None, (extract_id(f) for f in os.listdir(mask_d))))
        overlap = len(img_ids & mask_ids)
        if overlap > best_overlap:
            best_overlap = overlap
            best_pair = (img_d, mask_d)

image_dir, mask_dir = best_pair
print(f"Selected pair with {best_overlap} overlapping IDs:")
print("  image_dir:", image_dir)
print("  mask_dir:", mask_dir)
assert best_overlap > 0, "No overlapping IDs found — re-check folder structure."


In [ ]:
IMG_SIZE = 256

class FloodDataset(Dataset):
    def __init__(self, image_dir, mask_dir, subset_size=None, img_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size

        image_files = sorted(os.listdir(image_dir))
        mask_files = sorted(os.listdir(mask_dir))

        mask_by_id = {}
        for m in mask_files:
            mid = extract_id(m)
            if mid is not None:
                mask_by_id[mid] = m

        self.pairs = []
        for f in image_files:
            fid = extract_id(f)
            if fid is not None and fid in mask_by_id:
                self.pairs.append((f, mask_by_id[fid]))

        if subset_size is not None:
            self.pairs = self.pairs[:subset_size]

        print(f"Dataset initialized with {len(self.pairs)} matched image/mask pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        img = cv2.imread(os.path.join(self.image_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = torch.from_numpy(mask.astype(np.int64))

        return img, mask

# Free-tier sizing: 600 images total gives a real train/val split without an
# excessively long training run. Increase once you've confirmed everything works.
SUBSET_SIZE = 600
dataset = FloodDataset(image_dir, mask_dir, subset_size=SUBSET_SIZE, img_size=IMG_SIZE)
assert len(dataset) > 0, "Dataset has 0 pairs — check Section 2 output."


## 3. Confirm the water class ID from real data
Don't assume the class legend — compute the actual pixel distribution from a sample
of your masks and cross-check against FloodNet's published legend (commonly:
0=Background, 1=Building-flooded, 2=Building-non-flooded, 3=Road-flooded,
4=Road-non-flooded, 5=Water, 6=Tree, 7=Vehicle, 8=Pool, 9=Grass — verify this against
the dataset's own description/discussion tab on Kaggle, mirrors sometimes reindex).


In [ ]:
from collections import Counter

value_counts = Counter()
sample_n = min(30, len(dataset))
for i in range(sample_n):
    _, m = dataset[i]
    vals, counts = torch.unique(m, return_counts=True)
    for v, c in zip(vals.tolist(), counts.tolist()):
        value_counts[v] += c

total = sum(value_counts.values())
print(f"Class distribution across {sample_n} sample masks:")
print("Class ID : pixel count : percentage")
for cls_id, cnt in sorted(value_counts.items()):
    print(f"{cls_id:>3}      : {cnt:>10} : {100*cnt/total:.2f}%")

print("\nSet WATER_CLASS_ID below based on which class ID makes sense as 'water'")
print("(cross-check against FloodNet's legend on Kaggle before trusting this blindly).")


In [ ]:
# SET THIS based on the distribution printed above and FloodNet's documented legend
WATER_CLASS_ID = 5

def to_binary(mask_tensor, water_id=WATER_CLASS_ID):
    return (mask_tensor == water_id).long()

# quick check this produces a sane fraction (not 0%, not 100%)
_, test_mask = dataset[0]
frac = to_binary(test_mask).float().mean().item()
print(f"Fraction of water pixels in sample 0 with WATER_CLASS_ID={WATER_CLASS_ID}: {frac:.4f}")
if frac == 0.0:
    print("WARNING: 0% water pixels in this sample. Could be a dry image, or WATER_CLASS_ID could be wrong. Check a few more samples before proceeding.")


## 4. Train/val split
Same principle as NIDS: hold out data the model never trains on, to check it
generalizes rather than memorizes.


In [ ]:
val_fraction = 0.2
val_size = int(len(dataset) * val_fraction)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)   # fixed seed = reproducible split
)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type=="cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(device.type=="cuda"))


## 5. Loss function: Dice + BCE combined
BCE handles per-pixel classification. Dice directly rewards overlap between
predicted and true water regions, which matters more here since water is usually
a minority of pixels in any given image (class imbalance, same underlying issue as
imbalanced attack classes in tabular data).


In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        # targets: float tensor of 0/1, same shape as logits
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        smooth = 1e-6
        intersection = (probs * targets).sum(dim=(1,2,3))
        dice_score = (2. * intersection + smooth) / (probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + smooth)
        dice_loss = 1 - dice_score.mean()

        return bce_loss + dice_loss

criterion = DiceBCELoss()
print("Loss function ready.")


## 6. Model, optimizer, and IoU metric
Reusing the pretrained U-Net from Phase 1 — now we actually train it.
Mixed precision (`torch.cuda.amp`) is used on GPU to cut memory and speed up
training, which matters on free tier's shared/limited VRAM.


In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def compute_iou_batch(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    intersection = (preds * targets).sum(dim=(1,2,3))
    union = ((preds + targets) > 0).float().sum(dim=(1,2,3))
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()

print("Model, optimizer, and metric ready.")


## 7. Training loop with checkpointing
**Free-tier settings:** small epoch count (15) with early stopping if val IoU
plateaus, so you're not burning your session time budget on diminishing returns.
Best model (by val IoU) is saved to Drive after every improving epoch — safe against
a mid-training disconnect.


In [ ]:
NUM_EPOCHS = 15
PATIENCE = 4   # stop early if val IoU doesn't improve for this many epochs

best_val_iou = 0.0
epochs_without_improvement = 0
history = {"train_loss": [], "val_loss": [], "val_iou": []}

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    # --- training ---
    model.train()
    train_loss_total = 0.0
    for imgs_b, masks_b in train_loader:
        imgs_b = imgs_b.to(device)
        targets = to_binary(masks_b).unsqueeze(1).float().to(device)   # add channel dim

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(imgs_b)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_total += loss.item() * imgs_b.size(0)

    train_loss = train_loss_total / len(train_dataset)

    # --- validation ---
    model.eval()
    val_loss_total = 0.0
    val_iou_total = 0.0
    with torch.no_grad():
        for imgs_b, masks_b in val_loader:
            imgs_b = imgs_b.to(device)
            targets = to_binary(masks_b).unsqueeze(1).float().to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs_b)
                loss = criterion(logits, targets)

            val_loss_total += loss.item() * imgs_b.size(0)
            val_iou_total += compute_iou_batch(logits, targets) * imgs_b.size(0)

    val_loss = val_loss_total / len(val_dataset)
    val_iou = val_iou_total / len(val_dataset)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_iou={val_iou:.4f} | {elapsed:.1f}s")

    # checkpoint if improved
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        epochs_without_improvement = 0
        checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_iou": val_iou,
        }, checkpoint_path)
        print(f"  -> New best val_iou={val_iou:.4f}, checkpoint saved to Drive.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs, stopping early.")
            break

print(f"\nTraining complete. Best val IoU: {best_val_iou:.4f}")


## 8. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train loss")
axes[0].plot(history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over training")
axes[0].legend()

axes[1].plot(history["val_iou"], label="Val IoU", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].set_title("Validation IoU over training")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, "training_curves.png"))
plt.show()
print("Curves saved to Drive alongside the checkpoint — you'll want this image for your writeup.")


## 9. Load best checkpoint and do a final visual check
Compare this against the Phase 1 untrained-noise predictions — this is the actual
before/after evidence for your writeup.


In [ ]:
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"))
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {checkpoint['epoch']+1}, val_iou={checkpoint['val_iou']:.4f}")

imgs_b, masks_b = next(iter(val_loader))
imgs_b = imgs_b.to(device)
targets = to_binary(masks_b)

with torch.no_grad():
    logits = model(imgs_b)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i in range(min(4, imgs_b.shape[0])):
    axes[0, i].imshow(imgs_b[i].cpu().permute(1, 2, 0).numpy())
    axes[0, i].set_title(f"Input {i}")
    axes[1, i].imshow(targets[i].numpy(), cmap="Blues")
    axes[1, i].set_title(f"Ground truth {i}")
    axes[2, i].imshow(preds[i, 0].numpy(), cmap="Blues")
    axes[2, i].set_title(f"Trained prediction {i}")
    for row in range(3):
        axes[row, i].axis("off")
plt.tight_layout()
plt.show()

print(f"\nFinal baseline val IoU: {checkpoint['val_iou']:.4f}")
print("This is your Phase 2 reference number — Phase 3 (robustness testing) measures how much this drops under rain/fog/low-light.")


## Phase 2 complete
You now have:
- A trained model checkpoint saved to Drive (survives session disconnects)
- A real baseline IoU score (not the untrained noise from Phase 1)
- Training curves for your writeup
- Visual before/after comparison against Phase 1

**Next (Phase 3):** synthetically degrade validation images (rain/fog/low-light via
Albumentations) and measure how much this baseline IoU drops, then test whether
enhancement preprocessing recovers it. This is the core research contribution.
